# 01 - Load and explore

**Input:** `data/raw/match.csv`, `data/raw/set/<match>/set*.csv`
**What it does:** loads match-level metadata, prints basic counts (matches, players), builds the matchup-frequency table, counts set files per match, and peeks at one stroke-level file to confirm the schema.
**Output:** `data/processed/matchup_counts.csv`, `data/processed/set_files_per_match.csv`

In [1]:
## --- path bootstrap: run from the repo root no matter where this is launched ---
## nbconvert and some editors set the working directory to the notebook's own
## folder. Walk up until we find the repo root (the folder containing data/),
## chdir there so relative data paths resolve, and put code/ on sys.path so the
## shared modules (utils.py, shot_translations.py) import cleanly.
import os, sys
_d = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(_d, "data")):
        break
    _d = os.path.dirname(_d)
os.chdir(_d)
if os.path.join(_d, "code") not in sys.path:
    sys.path.insert(0, os.path.join(_d, "code"))
print("working directory:", os.getcwd())

working directory: /Users/aakankshvaidya/Desktop/qss20_final_project


In [2]:
import os
import pandas as pd

In [3]:
## CONFIG -- all paths relative to the repo root (no absolute paths)
RAW_DIR = "data/raw"
SET_DIR = os.path.join(RAW_DIR, "set")
PROC_DIR = "data/processed"
os.makedirs(PROC_DIR, exist_ok=True)

## Load match metadata

In [4]:
matches = pd.read_csv(os.path.join(RAW_DIR, "match.csv"))
print("=== match.csv loaded ===")
print("shape:", matches.shape)
print("columns:", list(matches.columns))
print()
print(matches.head())

=== match.csv loaded ===
shape: (58, 13)
columns: ['id', 'video', 'tournament', 'round', 'year', 'month', 'day', 'set', 'duration', 'winner', 'loser', 'downcourt', 'db']

   id                                              video  \
0   1  Ng_Ka_Long_Angus_Lee_Cheuk_Yiu_YONEX_Thailand_...   
1   2  Carolina_Marin_An_Se_Young_HSBC_BWF_WORLD_TOUR...   
2   3  Anthony_Sinisuka_Ginting_Lee_Zii_Jia_HSBC_BWF_...   
3   4  Ng_Ka_Long_Angus_Kidambi_Srikanth_HSBC_BWF_WOR...   
4   5  Pusarla_V._Sindhu_Pornpawee_Chochuwong_HSBC_BW...   

                        tournament           round    year  month   day  set  \
0         YONEX THAILAND OPEN 2021  Quarter-finals  2021.0    1.0  15.0    2   
1  HSBC BWF WORLD TOUR FINALS 2020  Quarter-finals  2021.0    1.0  29.0    3   
2  HSBC BWF WORLD TOUR FINALS 2020  Quarter-finals  2021.0    1.0  29.0    3   
3  HSBC BWF WORLD TOUR FINALS 2020  Quarter-finals  2021.0    1.0  29.0    3   
4  HSBC BWF WORLD TOUR FINALS 2020  Quarter-finals  2021.0    1.0  2

## Basic counts

In [5]:
n_matches = len(matches)
players = pd.unique(matches[["winner", "loser"]].values.ravel())
n_players = len(players)
print("number of matches:", n_matches)
print("number of unique players:", n_players)

number of matches: 58
number of unique players: 35


## Matchup frequency (which pairs play most often)

In [6]:
def matchup_pair(row):
    ## normalized (player1, player2) tuple where order doesn't matter
    a, b = row["winner"], row["loser"]
    return tuple(sorted([a, b]))

matches["matchup"] = matches.apply(matchup_pair, axis=1)
matchup_counts = matches["matchup"].value_counts().reset_index()
matchup_counts.columns = ["matchup", "n_matches"]
print("=== top matchups (most repeated) ===")
print(matchup_counts.head(10))

=== top matchups (most repeated) ===
                                        matchup  n_matches
0                (Akane YAMAGUCHI, An Se Young)          4
1                     (CHEN Yufei, HE Bingjiao)          3
2                 (Akane YAMAGUCHI, CHEN Yufei)          2
3               (CHEN Yufei, Ratchanok INTANON)          2
4             (LEE Cheuk Yiu, NG Ka Long Angus)          1
5                       (LEE Zii Jia, SHI Yuqi)          1
6            (Anders ANTONSEN, Kenta NISHIMOTO)          1
7  (Busanan ONGBAMRUNGPHAN, Supanida KATETHONG)          1
8                    (LEE Zii Jia, Lakshya SEN)          1
9          (Jonatan CHRISTIE, KIDAMBI Srikanth)          1


## Count set files per match

In [7]:
set_folders = [f for f in os.listdir(SET_DIR) if os.path.isdir(os.path.join(SET_DIR, f))]
print("number of match folders:", len(set_folders))

set_file_counts = []
for folder in set_folders:
    folder_path = os.path.join(SET_DIR, folder)
    csvs = [f for f in os.listdir(folder_path) if f.startswith("set") and f.endswith(".csv")]
    set_file_counts.append({"match_folder": folder, "n_set_files": len(csvs)})

set_files_df = pd.DataFrame(set_file_counts)
print("distribution of set files per match:")
print(set_files_df["n_set_files"].value_counts().sort_index())

number of match folders: 58
distribution of set files per match:
n_set_files
2    34
3    24
Name: count, dtype: int64


## Peek at one stroke-level file

In [8]:
sample_folder = set_folders[0]
sample_set_path = os.path.join(SET_DIR, sample_folder, "set1.csv")
sample_strokes = pd.read_csv(sample_set_path)
print("path:", sample_set_path)
print("shape:", sample_strokes.shape)
print("columns:", list(sample_strokes.columns))
print()
print(sample_strokes.head(3))

path: data/raw/set/Busanan_ONGBAMRUNGPHAN_Aakarshi_KASHYAP_YONEX_SUNRISE_India_Open_2022_Semifinals/set1.csv
shape: (496, 30)
columns: ['rally', 'ball_round', 'time', 'frame_num', 'roundscore_A', 'roundscore_B', 'player', 'server', 'type', 'aroundhead', 'backhand', 'hit_height', 'hit_area', 'hit_x', 'hit_y', 'landing_height', 'landing_area', 'landing_x', 'landing_y', 'lose_reason', 'win_reason', 'getpoint_player', 'flaw', 'player_location_area', 'player_location_x', 'player_location_y', 'opponent_location_area', 'opponent_location_x', 'opponent_location_y', 'db']

   rally  ball_round      time  frame_num  roundscore_A  roundscore_B player  \
0      1           1  00:07:01      12637             1             0      A   
1      1           2  00:07:03      12707             1             0      B   
2      1           3  00:07:04      12734             1             0      A   

   server type  aroundhead  ...  win_reason  getpoint_player  flaw  \
0       1  發長球         NaN  ...       

## Save summaries for later notebooks

In [9]:
matchup_counts.to_csv(os.path.join(PROC_DIR, "matchup_counts.csv"), index=False)
set_files_df.to_csv(os.path.join(PROC_DIR, "set_files_per_match.csv"), index=False)
print("saved matchup_counts.csv and set_files_per_match.csv")

saved matchup_counts.csv and set_files_per_match.csv
